# 모델 학습 코드 (xgboost, lightgbm, catboost, random forest, logistic)

## XGBoost

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
X_train_part = pd.read_csv('X_tr_unscaled.csv')
X_val_part = pd.read_csv('X_val_unscaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_unscaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'scale_pos_weight': scale_weight_opt,
        'random_state': 42,
        'n_jobs': -1,
        'tree_method': 'hist',
        'eval_metric': 'logloss'
    }
    model_opt = XGBClassifier(**params)
    model_opt.fit(X_opt_tr, y_opt_tr, eval_set=[(X_opt_val, y_opt_val)], verbose=False)
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 XGBoost 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 XGBoost K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

    # 🌟 Optuna가 위에서 찾아낸 최적의 조합을 덮어씌웁니다!
    model = XGBClassifier(
        **study.best_params,                    # 옵튜나 최적 파라미터 한방에 주입
        scale_pos_weight=scale_pos_weight_fold, # Fold별 불균형 비율 동적 적용
        random_state=42 + fold,                 # Fold마다 미세하게 다른 난수 부여
        n_jobs=-1,
        tree_method='hist',
        eval_metric='logloss'
    )
    
    # 조기 종료(Early Stopping)를 적용하여 과적합 방지
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )
    
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


📥 데이터를 불러오고 K-Fold를 위해 병합합니다...
✅ 병합 완료! (전체 학습 데이터 X: (360000, 44), y: (360000,))


c:\Users\kym92\.conda\envs\yu-m-n-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...


[I 2026-04-09 14:47:37,935] A new study created in memory with name: no-name-80f92389-588c-4ff5-82bd-77bb54a18be5
[I 2026-04-09 14:48:10,683] Trial 0 finished with value: 0.6916570911723228 and parameters: {'n_estimators': 649, 'max_depth': 7, 'learning_rate': 0.04185436140946885, 'subsample': 0.7991696048221746, 'colsample_bytree': 0.6201993441029783, 'gamma': 0.5950712578963105, 'reg_lambda': 1.344906795553105}. Best is trial 0 with value: 0.6916570911723228.
[I 2026-04-09 14:48:56,669] Trial 1 finished with value: 0.873477934180568 and parameters: {'n_estimators': 653, 'max_depth': 10, 'learning_rate': 0.060289240354963115, 'subsample': 0.7662576201280478, 'colsample_bytree': 0.6578710673671259, 'gamma': 0.4532463374369916, 'reg_lambda': 3.064711325483229}. Best is trial 1 with value: 0.873477934180568.
[I 2026-04-09 14:49:29,201] Trial 2 finished with value: 0.7269849644362776 and parameters: {'n_estimators': 490, 'max_depth': 9, 'learning_rate': 0.02622135837976295, 'subsample': 0

🎉 탐색 완료! 최고 Validation AUC: 0.9125

🚀 XGBoost K-Fold 학습을 시작합니다...
   - Fold 1 완료 (ROC-AUC: 0.9114)
   - Fold 2 완료 (ROC-AUC: 0.9162)
   - Fold 3 완료 (ROC-AUC: 0.9142)
   - Fold 4 완료 (ROC-AUC: 0.9132)
   - Fold 5 완료 (ROC-AUC: 0.9126)

--- 📊 최종 Validation(OOF) 결과 ---
정확도 (Accuracy): 0.8446
F1 점수 (F1 Score): 0.8356
AUC (ROC-AUC):  0.9135
Fold별 AUC: [0.9114, 0.9162, 0.9142, 0.9132, 0.9126]


### XGBoost CSV 추출

In [2]:
import numpy as np
import pandas as pd

model_name = "XGBoost"
model_suffix = "xgboost"

print(f"\n💾 {model_name} 대시보드/제출용 CSV 파일 추출을 시작합니다...")

# validation 결과는 OOF 전체가 아니라, 원래 validation 파트에 해당하는 마지막 구간만 저장
n_val = len(X_val_part)
val_pred_prob = np.asarray(oof_pred)[-n_val:]
val_pred_label = np.asarray(oof_label)[-n_val:]

val_df = pd.DataFrame({
    "pred_prob": val_pred_prob,
    "pred_label": val_pred_label,
    "returned": y_val_part.values
})
val_df.to_csv(f"val_predictions_{model_suffix}.csv", index=False)
print(f"✅ val_predictions_{model_suffix}.csv 생성 완료! (rows={len(val_df)})")

test_label = (np.asarray(test_pred) >= 0.5).astype(int)
test_df = pd.DataFrame({
    "order_id": test_id_df["order_id"],
    "pred_prob": np.asarray(test_pred),
    "pred_label": test_label
})
test_df.to_csv(f"test_predictions_{model_suffix}.csv", index=False)
print(f"✅ test_predictions_{model_suffix}.csv 생성 완료! (rows={len(test_df)})")

# 모델별 변수 중요도 저장
if hasattr(model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": model.feature_importances_
    }).sort_values(by="importance", ascending=False)
    importance_df.to_csv(f"feature_importance_{model_suffix}.csv", index=False)
    print(f"✅ feature_importance_{model_suffix}.csv 생성 완료!")
elif hasattr(model, "coef_"):
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": np.abs(np.ravel(model.coef_))
    }).sort_values(by="importance", ascending=False)
    importance_df.to_csv(f"feature_importance_{model_suffix}.csv", index=False)
    print(f"✅ feature_importance_{model_suffix}.csv 생성 완료! (coef 절댓값 기준)")
else:
    print(f"⚠️ {model_name}은 기본 feature importance를 제공하지 않아 저장을 생략합니다.")


💾 XGBoost 대시보드/제출용 CSV 파일 추출을 시작합니다...
✅ val_predictions_xgboost.csv 생성 완료! (rows=200000)
✅ test_predictions_xgboost.csv 생성 완료! (rows=50000)
✅ feature_importance_xgboost.csv 생성 완료!


## Lightgbm

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from lightgbm import LGBMClassifier

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
X_train_part = pd.read_csv('X_tr_unscaled.csv')
X_val_part = pd.read_csv('X_val_unscaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_unscaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    # LightGBM 전용 과적합 방지 로직
    if params['num_leaves'] > (2 ** params['max_depth']):
        params['num_leaves'] = (2 ** params['max_depth']) - 1

    model_opt = LGBMClassifier(**params)
    model_opt.fit(X_opt_tr, y_opt_tr, eval_set=[(X_opt_val, y_opt_val)])
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 lightGBM 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 lightGMB K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

    # 🌟 Optuna가 위에서 찾아낸 최적의 조합을 덮어씌웁니다!
    model = LGBMClassifier(
        **study.best_params,        # 옵튜나 최적 파라미터 한방에 주입
        class_weight='balanced',    # LightGBM용 불균형 처리 설정
        random_state=42 + fold,     # Fold마다 미세하게 다른 난수 부여
        n_jobs=-1,
        verbose=-1
    )    
    # 조기 종료(Early Stopping)를 적용하여 과적합 방지
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
    )
    
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


📥 데이터를 불러오고 K-Fold를 위해 병합합니다...
✅ 병합 완료! (전체 학습 데이터 X: (360000, 44), y: (360000,))

🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...


[I 2026-04-09 15:15:09,124] A new study created in memory with name: no-name-14cb6760-93ca-4813-927b-33b09e34bd42
[I 2026-04-09 15:15:30,158] Trial 0 finished with value: 0.6420478346763274 and parameters: {'n_estimators': 735, 'max_depth': 9, 'learning_rate': 0.019851073467393406, 'num_leaves': 60, 'subsample': 0.7632854505455715, 'colsample_bytree': 0.8471726885596815}. Best is trial 0 with value: 0.6420478346763274.
[I 2026-04-09 15:15:47,194] Trial 1 finished with value: 0.635371350533975 and parameters: {'n_estimators': 367, 'max_depth': 9, 'learning_rate': 0.015773679254870884, 'num_leaves': 122, 'subsample': 0.9794021877789211, 'colsample_bytree': 0.9789955748401272}. Best is trial 0 with value: 0.6420478346763274.
[I 2026-04-09 15:16:08,536] Trial 2 finished with value: 0.6595489360666948 and parameters: {'n_estimators': 831, 'max_depth': 6, 'learning_rate': 0.039033328909812245, 'num_leaves': 84, 'subsample': 0.6489896776576235, 'colsample_bytree': 0.9761892595308678}. Best is

🎉 탐색 완료! 최고 Validation AUC: 0.8185

🚀 lightGMB K-Fold 학습을 시작합니다...
   - Fold 1 완료 (ROC-AUC: 0.8178)
   - Fold 2 완료 (ROC-AUC: 0.8243)
   - Fold 3 완료 (ROC-AUC: 0.8199)
   - Fold 4 완료 (ROC-AUC: 0.8206)
   - Fold 5 완료 (ROC-AUC: 0.8209)

--- 📊 최종 Validation(OOF) 결과 ---
정확도 (Accuracy): 0.7458
F1 점수 (F1 Score): 0.7326
AUC (ROC-AUC):  0.8207
Fold별 AUC: [0.8178, 0.8243, 0.8199, 0.8206, 0.8209]


### LightGBM CSV 추출

In [4]:
import numpy as np
import pandas as pd

model_name = "LightGBM"
model_suffix = "lightgbm"

print(f"\n💾 {model_name} 대시보드/제출용 CSV 파일 추출을 시작합니다...")

# validation 결과는 OOF 전체가 아니라, 원래 validation 파트에 해당하는 마지막 구간만 저장
n_val = len(X_val_part)
val_pred_prob = np.asarray(oof_pred)[-n_val:]
val_pred_label = np.asarray(oof_label)[-n_val:]

val_df = pd.DataFrame({
    "pred_prob": val_pred_prob,
    "pred_label": val_pred_label,
    "returned": y_val_part.values
})
val_df.to_csv(f"val_predictions_{model_suffix}.csv", index=False)
print(f"✅ val_predictions_{model_suffix}.csv 생성 완료! (rows={len(val_df)})")

test_label = (np.asarray(test_pred) >= 0.5).astype(int)
test_df = pd.DataFrame({
    "order_id": test_id_df["order_id"],
    "pred_prob": np.asarray(test_pred),
    "pred_label": test_label
})
test_df.to_csv(f"test_predictions_{model_suffix}.csv", index=False)
print(f"✅ test_predictions_{model_suffix}.csv 생성 완료! (rows={len(test_df)})")

# 모델별 변수 중요도 저장
if hasattr(model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": model.feature_importances_
    }).sort_values(by="importance", ascending=False)
    importance_df.to_csv(f"feature_importance_{model_suffix}.csv", index=False)
    print(f"✅ feature_importance_{model_suffix}.csv 생성 완료!")
elif hasattr(model, "coef_"):
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": np.abs(np.ravel(model.coef_))
    }).sort_values(by="importance", ascending=False)
    importance_df.to_csv(f"feature_importance_{model_suffix}.csv", index=False)
    print(f"✅ feature_importance_{model_suffix}.csv 생성 완료! (coef 절댓값 기준)")
else:
    print(f"⚠️ {model_name}은 기본 feature importance를 제공하지 않아 저장을 생략합니다.")


💾 LightGBM 대시보드/제출용 CSV 파일 추출을 시작합니다...
✅ val_predictions_lightgbm.csv 생성 완료! (rows=200000)
✅ test_predictions_lightgbm.csv 생성 완료! (rows=50000)
✅ feature_importance_lightgbm.csv 생성 완료!


## Catboost

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from catboost import CatBoostClassifier

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
X_train_part = pd.read_csv('X_tr_unscaled.csv')
X_val_part = pd.read_csv('X_val_unscaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_unscaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'auto_class_weights': 'Balanced',
        'random_state': 42,
        'verbose': 0  # 🌟 추가: CatBoost는 0으로 해야 로그가 숨겨집니다.
    }
    
    model_opt = CatBoostClassifier(**params)
    
    # verbose 관련 에러 방지를 위해 여기서는 eval_set만 남깁니다.
    model_opt.fit(X_opt_tr, y_opt_tr, eval_set=[(X_opt_val, y_opt_val)])
    
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 XGBoost 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 CatBoost K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

    # 🌟 Optuna 결과 덮어쓰기
    model = CatBoostClassifier(
        **study.best_params,
        random_state=42 + fold,     # Fold마다 미세하게 다른 난수 부여
        verbose=0                   # 🌟 추가: 로그 숨김
    )
    
    # 🌟 fit 수정: verbose 관련 내용 제거
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])
    
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


📥 데이터를 불러오고 K-Fold를 위해 병합합니다...
✅ 병합 완료! (전체 학습 데이터 X: (360000, 44), y: (360000,))

🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...


[I 2026-04-09 15:28:18,812] A new study created in memory with name: no-name-f4cdad9f-f7f4-4abd-b962-e910a36f1feb
[I 2026-04-09 15:28:55,716] Trial 0 finished with value: 0.6053538692770171 and parameters: {'iterations': 452, 'depth': 8, 'learning_rate': 0.010694037839734067, 'l2_leaf_reg': 2.3441704312241463}. Best is trial 0 with value: 0.6053538692770171.
[I 2026-04-09 15:30:08,575] Trial 1 finished with value: 0.6373497357000022 and parameters: {'iterations': 955, 'depth': 8, 'learning_rate': 0.024744645040504037, 'l2_leaf_reg': 4.057305377790986}. Best is trial 1 with value: 0.6373497357000022.
[I 2026-04-09 15:30:42,228] Trial 2 finished with value: 0.6341029438007895 and parameters: {'iterations': 529, 'depth': 7, 'learning_rate': 0.08360520370959518, 'l2_leaf_reg': 9.530343730170417}. Best is trial 1 with value: 0.6373497357000022.
[I 2026-04-09 15:31:12,662] Trial 3 finished with value: 0.618239007311 and parameters: {'iterations': 422, 'depth': 7, 'learning_rate': 0.052112466

### CatBoost CSV 추출

In [ ]:
import numpy as np
import pandas as pd

model_name = "CatBoost"
model_suffix = "catb"

print(f"\n💾 {model_name} 대시보드/제출용 CSV 파일 추출을 시작합니다...")

# validation 결과는 OOF 전체가 아니라, 원래 validation 파트에 해당하는 마지막 구간만 저장
n_val = len(X_val_part)
val_pred_prob = np.asarray(oof_pred)[-n_val:]
val_pred_label = np.asarray(oof_label)[-n_val:]

val_df = pd.DataFrame({
    "pred_prob": val_pred_prob,
    "pred_label": val_pred_label,
    "returned": y_val_part.values
})
val_df.to_csv(f"val_predictions_{model_suffix}.csv", index=False)
print(f"✅ val_predictions_{model_suffix}.csv 생성 완료! (rows={len(val_df)})")

test_label = (np.asarray(test_pred) >= 0.5).astype(int)
test_df = pd.DataFrame({
    "order_id": test_id_df["order_id"],
    "pred_prob": np.asarray(test_pred),
    "pred_label": test_label
})
test_df.to_csv(f"test_predictions_{model_suffix}.csv", index=False)
print(f"✅ test_predictions_{model_suffix}.csv 생성 완료! (rows={len(test_df)})")

# 모델별 변수 중요도 저장
if hasattr(model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": model.feature_importances_
    }).sort_values(by="importance", ascending=False)
    importance_df.to_csv(f"feature_importance_{model_suffix}.csv", index=False)
    print(f"✅ feature_importance_{model_suffix}.csv 생성 완료!")
elif hasattr(model, "coef_"):
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": np.abs(np.ravel(model.coef_))
    }).sort_values(by="importance", ascending=False)
    importance_df.to_csv(f"feature_importance_{model_suffix}.csv", index=False)
    print(f"✅ feature_importance_{model_suffix}.csv 생성 완료! (coef 절댓값 기준)")
else:
    print(f"⚠️ {model_name}은 기본 feature importance를 제공하지 않아 저장을 생략합니다.")

## RandomForest

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
X_train_part = pd.read_csv('X_tr_unscaled.csv')
X_val_part = pd.read_csv('X_val_unscaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_unscaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 10, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1
    }
    
    model_opt = RandomForestClassifier(**params)
    
    # 🚨 주의: Random Forest는 eval_set이 없으므로 깔끔하게 X와 y만 넣습니다!
    model_opt.fit(X_opt_tr, y_opt_tr)
    
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 XGBoost 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 XGBoost K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

# 🌟 Optuna 결과 덮어쓰기
    model = RandomForestClassifier(
        **study.best_params,
        class_weight='balanced',    # RF용 불균형 처리
        random_state=42 + fold,     
        n_jobs=-1,
    )
    
    # 🚨 주의: 여기서도 eval_set을 완전히 제거해야 에러가 나지 않습니다!
    model.fit(X_tr, y_tr)
    
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


### RandomForest CSV 추출

In [ ]:
import numpy as np
import pandas as pd

model_name = "RandomForest"
model_suffix = "rf"

print(f"\n💾 {model_name} 대시보드/제출용 CSV 파일 추출을 시작합니다...")

# validation 결과는 OOF 전체가 아니라, 원래 validation 파트에 해당하는 마지막 구간만 저장
n_val = len(X_val_part)
val_pred_prob = np.asarray(oof_pred)[-n_val:]
val_pred_label = np.asarray(oof_label)[-n_val:]

val_df = pd.DataFrame({
    "pred_prob": val_pred_prob,
    "pred_label": val_pred_label,
    "returned": y_val_part.values
})
val_df.to_csv(f"val_predictions_{model_suffix}.csv", index=False)
print(f"✅ val_predictions_{model_suffix}.csv 생성 완료! (rows={len(val_df)})")

test_label = (np.asarray(test_pred) >= 0.5).astype(int)
test_df = pd.DataFrame({
    "order_id": test_id_df["order_id"],
    "pred_prob": np.asarray(test_pred),
    "pred_label": test_label
})
test_df.to_csv(f"test_predictions_{model_suffix}.csv", index=False)
print(f"✅ test_predictions_{model_suffix}.csv 생성 완료! (rows={len(test_df)})")

# 모델별 변수 중요도 저장
if hasattr(model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": model.feature_importances_
    }).sort_values(by="importance", ascending=False)
    importance_df.to_csv(f"feature_importance_{model_suffix}.csv", index=False)
    print(f"✅ feature_importance_{model_suffix}.csv 생성 완료!")
elif hasattr(model, "coef_"):
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": np.abs(np.ravel(model.coef_))
    }).sort_values(by="importance", ascending=False)
    importance_df.to_csv(f"feature_importance_{model_suffix}.csv", index=False)
    print(f"✅ feature_importance_{model_suffix}.csv 생성 완료! (coef 절댓값 기준)")
else:
    print(f"⚠️ {model_name}은 기본 feature importance를 제공하지 않아 저장을 생략합니다.")

## Logistic

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
# 🚨 반드시 스케일링된 데이터를 불러와야 합니다!
X_train_part = pd.read_csv('X_tr_scaled.csv')
X_val_part = pd.read_csv('X_val_scaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_scaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        # C값이 작을수록 규제가 강해짐 (과적합 방지)
        'C': trial.suggest_float('C', 0.001, 10.0, log=True), 
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear']),
        'class_weight': 'balanced',
        'max_iter': 1000, # 에러 방지를 위해 반복 횟수를 넉넉히 줍니다
        'random_state': 42
    }
    
    model_opt = LogisticRegression(**params)
    
    # 🚨 로지스틱은 eval_set이 없습니다.
    model_opt.fit(X_opt_tr, y_opt_tr)
    
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 XGBoost 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 Logistic K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

# 🌟 Optuna 결과 덮어쓰기
    model = LogisticRegression(
        **study.best_params,
        class_weight='balanced',
        random_state=42 + fold,
        max_iter=1000
    )
    
    # 🚨 eval_set 제거
    model.fit(X_tr, y_tr)
    
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


### Logistic Regression CSV 추출

In [ ]:
import numpy as np
import pandas as pd

model_name = "Logistic Regression"
model_suffix = "logistic"

print(f"\n💾 {model_name} 대시보드/제출용 CSV 파일 추출을 시작합니다...")

# validation 결과는 OOF 전체가 아니라, 원래 validation 파트에 해당하는 마지막 구간만 저장
n_val = len(X_val_part)
val_pred_prob = np.asarray(oof_pred)[-n_val:]
val_pred_label = np.asarray(oof_label)[-n_val:]

val_df = pd.DataFrame({
    "pred_prob": val_pred_prob,
    "pred_label": val_pred_label,
    "returned": y_val_part.values
})
val_df.to_csv(f"val_predictions_{model_suffix}.csv", index=False)
print(f"✅ val_predictions_{model_suffix}.csv 생성 완료! (rows={len(val_df)})")

test_label = (np.asarray(test_pred) >= 0.5).astype(int)
test_df = pd.DataFrame({
    "order_id": test_id_df["order_id"],
    "pred_prob": np.asarray(test_pred),
    "pred_label": test_label
})
test_df.to_csv(f"test_predictions_{model_suffix}.csv", index=False)
print(f"✅ test_predictions_{model_suffix}.csv 생성 완료! (rows={len(test_df)})")

# 모델별 변수 중요도 저장
if hasattr(model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": model.feature_importances_
    }).sort_values(by="importance", ascending=False)
    importance_df.to_csv(f"feature_importance_{model_suffix}.csv", index=False)
    print(f"✅ feature_importance_{model_suffix}.csv 생성 완료!")
elif hasattr(model, "coef_"):
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": np.abs(np.ravel(model.coef_))
    }).sort_values(by="importance", ascending=False)
    importance_df.to_csv(f"feature_importance_{model_suffix}.csv", index=False)
    print(f"✅ feature_importance_{model_suffix}.csv 생성 완료! (coef 절댓값 기준)")
else:
    print(f"⚠️ {model_name}은 기본 feature importance를 제공하지 않아 저장을 생략합니다.")

# KNN

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
X_train_part = pd.read_csv('X_tr_scaled.csv')
X_val_part = pd.read_csv('X_val_scaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_scaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        'n_neighbors': trial.suggest_int('n_neighbors', 5, 50),
        'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
        'p': trial.suggest_categorical('p', [1, 2]) # 1: 맨해튼 거리, 2: 유클리디안 거리
    }
    model_opt = KNeighborsClassifier(**params)
    model_opt.fit(X_opt_tr, y_opt_tr)
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 XGBoost 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 XGBoost K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

    # 🌟 Optuna 결과 덮어쓰기
    model = KNeighborsClassifier(
        **study.best_params,
        n_jobs=-1
        # 🚨 주의: KNN은 random_state나 class_weight 파라미터가 없으므로 지워야 에러가 안 납니다!
    )
    
    # 🚨 주의: eval_set 없음
    model.fit(X_tr, y_tr)
    
    
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


### KNN CSV 추출

In [ ]:
import numpy as np
import pandas as pd

model_name = "KNN"
model_suffix = "knn"

print(f"\n💾 {model_name} 대시보드/제출용 CSV 파일 추출을 시작합니다...")

# validation 결과는 OOF 전체가 아니라, 원래 validation 파트에 해당하는 마지막 구간만 저장
n_val = len(X_val_part)
val_pred_prob = np.asarray(oof_pred)[-n_val:]
val_pred_label = np.asarray(oof_label)[-n_val:]

val_df = pd.DataFrame({
    "pred_prob": val_pred_prob,
    "pred_label": val_pred_label,
    "returned": y_val_part.values
})
val_df.to_csv(f"val_predictions_{model_suffix}.csv", index=False)
print(f"✅ val_predictions_{model_suffix}.csv 생성 완료! (rows={len(val_df)})")

test_label = (np.asarray(test_pred) >= 0.5).astype(int)
test_df = pd.DataFrame({
    "order_id": test_id_df["order_id"],
    "pred_prob": np.asarray(test_pred),
    "pred_label": test_label
})
test_df.to_csv(f"test_predictions_{model_suffix}.csv", index=False)
print(f"✅ test_predictions_{model_suffix}.csv 생성 완료! (rows={len(test_df)})")

# 모델별 변수 중요도 저장
if hasattr(model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": model.feature_importances_
    }).sort_values(by="importance", ascending=False)
    importance_df.to_csv(f"feature_importance_{model_suffix}.csv", index=False)
    print(f"✅ feature_importance_{model_suffix}.csv 생성 완료!")
elif hasattr(model, "coef_"):
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": np.abs(np.ravel(model.coef_))
    }).sort_values(by="importance", ascending=False)
    importance_df.to_csv(f"feature_importance_{model_suffix}.csv", index=False)
    print(f"✅ feature_importance_{model_suffix}.csv 생성 완료! (coef 절댓값 기준)")
else:
    print(f"⚠️ {model_name}은 기본 feature importance를 제공하지 않아 저장을 생략합니다.")

# ExTrees

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.ensemble import ExtraTreesClassifier

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
X_train_part = pd.read_csv('X_tr_unscaled.csv')
X_val_part = pd.read_csv('X_val_unscaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_unscaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 10, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1
    }
    
    model_opt = ExtraTreesClassifier(**params)
    
    # 🚨 주의: eval_set이 없으므로 통째로 뺍니다!
    model_opt.fit(X_opt_tr, y_opt_tr)
    
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 XGBoost 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 XGBoost K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

# 🌟 Optuna 결과 덮어쓰기
    model = ExtraTreesClassifier(
        **study.best_params,
        class_weight='balanced',
        random_state=42 + fold,
        n_jobs=-1
    )
    
    # 🚨 주의: 여기서도 eval_set을 뺍니다!
    model.fit(X_tr, y_tr)
    
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


### ExtraTrees CSV 추출

In [ ]:
import numpy as np
import pandas as pd

model_name = "ExtraTrees"
model_suffix = "extrees"

print(f"\n💾 {model_name} 대시보드/제출용 CSV 파일 추출을 시작합니다...")

# validation 결과는 OOF 전체가 아니라, 원래 validation 파트에 해당하는 마지막 구간만 저장
n_val = len(X_val_part)
val_pred_prob = np.asarray(oof_pred)[-n_val:]
val_pred_label = np.asarray(oof_label)[-n_val:]

val_df = pd.DataFrame({
    "pred_prob": val_pred_prob,
    "pred_label": val_pred_label,
    "returned": y_val_part.values
})
val_df.to_csv(f"val_predictions_{model_suffix}.csv", index=False)
print(f"✅ val_predictions_{model_suffix}.csv 생성 완료! (rows={len(val_df)})")

test_label = (np.asarray(test_pred) >= 0.5).astype(int)
test_df = pd.DataFrame({
    "order_id": test_id_df["order_id"],
    "pred_prob": np.asarray(test_pred),
    "pred_label": test_label
})
test_df.to_csv(f"test_predictions_{model_suffix}.csv", index=False)
print(f"✅ test_predictions_{model_suffix}.csv 생성 완료! (rows={len(test_df)})")

# 모델별 변수 중요도 저장
if hasattr(model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": model.feature_importances_
    }).sort_values(by="importance", ascending=False)
    importance_df.to_csv(f"feature_importance_{model_suffix}.csv", index=False)
    print(f"✅ feature_importance_{model_suffix}.csv 생성 완료!")
elif hasattr(model, "coef_"):
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": np.abs(np.ravel(model.coef_))
    }).sort_values(by="importance", ascending=False)
    importance_df.to_csv(f"feature_importance_{model_suffix}.csv", index=False)
    print(f"✅ feature_importance_{model_suffix}.csv 생성 완료! (coef 절댓값 기준)")
else:
    print(f"⚠️ {model_name}은 기본 feature importance를 제공하지 않아 저장을 생략합니다.")